In [ ]:
!pip install --upgrade huggingface_hub ipywidgets -q


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers accelerate
!pip install torchvision
!pip install -r requirements.txt

In [ ]:
import sys
!{sys.executable} -m pip install pyarrow -q

import pandas as pd

splits = {'test': 'professional_law/test-00000-of-00001.parquet', 'validation': 'professional_law/validation-00000-of-00001.parquet', 'dev': 'professional_law/dev-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/cais/mmlu/" + splits["test"])
df.to_csv('law_data_mc.csv')

In [ ]:
import ast
import sys
import pandas as pd

LABELS = ["A", "B", "C", "D"]

def transform(input_path, output_path):
    df = pd.read_csv(input_path, index_col=0)

    rows = []
    for i, row in df.iterrows():
        choices = row["choices"][1:-1].split('\'')
        rows.append({
            "id":       i,
            "question": row["question"],
            "choice_0": choices[0],
            "choice_1": choices[1],
            "choice_2": choices[2],
            "choice_3": choices[3],
            "answer":   LABELS[int(row["answer"])] if str(row["answer"]) not in LABELS else row["answer"],
        })

    out = pd.DataFrame(rows)
    out.to_csv(output_path, index=False)
    print(f"Saved {len(out)} rows → {output_path}")

transform("law_data_mc.csv", "law_data_cleaned.csv")

In [ ]:
# Model Inference Boilerplate — Gemma 4 (E2B / E2B-it)
#
# Loads Google's `gemma-4-E2B` (base) and `gemma-4-E2B-it` (instruction-tuned)
# models and runs text generation on both, so you can compare their behavior.

# 1. Import Libraries
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

# 2. Load Models and Processors
# Loads both the base (pre-trained) and instruction-tuned variants.
BASE_MODEL_ID = "google/gemma-4-E2B"
IT_MODEL_ID = "google/gemma-4-E2B-it"

base_processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype="auto",
    device_map="auto",
)

it_processor = AutoProcessor.from_pretrained(IT_MODEL_ID)
it_model = AutoModelForCausalLM.from_pretrained(
    IT_MODEL_ID,
    dtype="auto",
    device_map="auto",
)

# 3. Define Input
prompt = "Write a short joke about saving RAM."

# 4. Base Model (Non-Instruction-Tuned) Inference
# The base model does plain text completion — no chat template.
inputs = base_processor(text=prompt, return_tensors="pt").to(base_model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = base_model.generate(**inputs, max_new_tokens=256)
base_response = base_processor.decode(outputs[0][input_len:], skip_special_tokens=True)
print(base_response)

# 5. Instruction-Tuned Model Inference
# The `-it` model expects chat-formatted messages via `apply_chat_template`.
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt},
]

text = it_processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
inputs = it_processor(text=text, return_tensors="pt").to(it_model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = it_model.generate(**inputs, max_new_tokens=256)
response = it_processor.decode(outputs[0][input_len:], skip_special_tokens=False)
it_response = it_processor.parse_response(response)
print(it_response)

# 6. Compare Outputs
print("Base model:\n", base_response)
print("\nInstruction-tuned model:\n", it_response)

In [ ]:
import ast
import os
import re
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForCausalLM

# ── 1. Load models ────────────────────────────────────────────────────────────
BASE_MODEL_ID = "google/gemma-4-E2B"
IT_MODEL_ID   = "google/gemma-4-E2B-it"

base_processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, dtype="auto", device_map="auto")

it_processor = AutoProcessor.from_pretrained(IT_MODEL_ID)
it_model = AutoModelForCausalLM.from_pretrained(IT_MODEL_ID, dtype="auto", device_map="auto")

# ── 2. Load data ──────────────────────────────────────────────────────────────
df = pd.read_csv("subset_100_en.csv")

# Convert numeric answer to letter (0-indexed: 0=A, 1=B, 2=C, 3=D)
LABELS = ["A", "B", "C", "D"]
df["answer"] = df["answer"].apply(lambda x: LABELS[int(x)] if str(x) not in LABELS else x)

OUTPUT_CSV = "english_results.csv"

if os.path.exists(OUTPUT_CSV):
    done = pd.read_csv(OUTPUT_CSV)
    start_idx = len(done)
    print(f"Resuming from row {start_idx}")
else:
    start_idx = 0
    pd.DataFrame(columns=[
    "id", "question", "choice_0", "choice_1", "choice_2", "choice_3", "answer",
    "gemma", "gemma_conf",
    "gemma_it", "gemma_it_conf",
]).to_csv(OUTPUT_CSV, index=False)

# ── 3. Helpers ────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are a helpful assistant answering multiple-choice law questions. "
    "Always respond on a single line in exactly this format: LETTER, CONFIDENCE\n"
    "where LETTER is A, B, C, or D and CONFIDENCE is your confidence from 1 (guessing) to 10 (certain).\n"
    "Example: C, 7\n"
    "Do not include any other text."
)

def format_prompt(question, choices):
    options = "\n".join(f"{LABELS[i]}) {c}" for i, c in enumerate(choices))
    return f"Question: {question}\n{options}\nAnswer:"

def parse_response(text):
    """Extract (letter, confidence) from model output. Returns ('?', None) on failure."""
    text = text.strip().upper()
    match = re.search(r'\b([A-D])\b.*?(\d+)', text)
    if match:
        letter = match.group(1)
        conf = min(10, max(1, int(match.group(2))))
        return letter, conf
    # Fallback: at least grab the letter
    letter_match = re.search(r'\b([A-D])\b', text)
    return (letter_match.group(1) if letter_match else "?"), None

def run_base(question, choices):
    prompt = format_prompt(question, choices)
    # Prepend system prompt as plain text for base model (no chat template)
    full_prompt = SYSTEM_PROMPT + "\n\n" + prompt
    inputs = base_processor(text=full_prompt, return_tensors="pt").to(base_model.device)
    input_len = inputs["input_ids"].shape[-1]
    outputs = base_model.generate(**inputs, max_new_tokens=16)
    raw = base_processor.decode(outputs[0][input_len:], skip_special_tokens=True)
    return parse_response(raw)

def run_it(question, choices):
    prompt = format_prompt(question, choices)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    text = it_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = it_processor(text=text, return_tensors="pt").to(it_model.device)
    input_len = inputs["input_ids"].shape[-1]
    outputs = it_model.generate(**inputs, max_new_tokens=16)
    raw = it_processor.decode(outputs[0][input_len:], skip_special_tokens=False)
    parsed = it_processor.parse_response(raw)
    content = parsed["content"] if isinstance(parsed, dict) else str(parsed)
    return parse_response(content)

# ── 4. Run inference ──────────────────────────────────────────────────────────
for i, row in tqdm(df.iloc[start_idx:].iterrows(), total=len(df) - start_idx, desc="Inference"):
    choices = [row["choice_0"], row["choice_1"], row["choice_2"], row["choice_3"]]

    gemma,    gemma_conf    = run_base(row["question"], choices)
    gemma_it, gemma_it_conf = run_it(row["question"],   choices)

    result = pd.DataFrame([{
        "id":            row["id"],
        "question":      row["question"],
        "choice_0":      row["choice_0"],
        "choice_1":      row["choice_1"],
        "choice_2":      row["choice_2"],
        "choice_3":      row["choice_3"],
        "answer":        row["answer"],
        "gemma":         gemma,
        "gemma_conf":    gemma_conf,
        "gemma_it":      gemma_it,
        "gemma_it_conf": gemma_it_conf,
    }])
    result.to_csv(OUTPUT_CSV, mode="a", header=False, index=False)

print("Done →", OUTPUT_CSV)

In [1]:
import os
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForCausalLM

# ── 1. Load models ────────────────────────────────────────────────────────────
BASE_MODEL_ID = "google/gemma-4-E2B"
IT_MODEL_ID   = "google/gemma-4-E2B-it"

base_processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, dtype="auto", device_map="auto")

it_processor = AutoProcessor.from_pretrained(IT_MODEL_ID)
it_model = AutoModelForCausalLM.from_pretrained(IT_MODEL_ID, dtype="auto", device_map="auto")

# ── 2. Load data ──────────────────────────────────────────────────────────────
INPUT_CSV  = "law_data_mc_FIXED.csv"
OUTPUT_CSV = "FIXED_full_law_results.csv"

df = pd.read_csv(INPUT_CSV)

LABELS = ["A", "B", "C", "D"]
# Ground-truth: numeric 0-3 -> letter. (If already a letter, leave it.)
df["answer"] = df["answer"].apply(lambda x: LABELS[int(x)] if str(x) not in LABELS else x)

# ── 3. Resolve the A/B/C/D token ids for each tokenizer ────────────────────────
# We score the *first answer token* by its logit, so we need the exact token id
# the model would emit for each letter in this context. Encoding "Answer: A" and
# taking the last token gives the "▁A"-style token that actually follows a colon,
# which is what generation would produce. Computed per-tokenizer to be safe.
def get_letter_token_ids(processor):
    tok = getattr(processor, "tokenizer", processor)
    ids = []
    for L in LABELS:
        enc = tok.encode("Answer: " + L, add_special_tokens=False)
        ids.append(enc[-1])
    assert len(set(ids)) == len(LABELS), f"Letter tokens not distinct: {ids} — inspect tokenizer."
    return ids

BASE_LETTER_IDS = get_letter_token_ids(base_processor)
IT_LETTER_IDS   = get_letter_token_ids(it_processor)

# ── 4. Prompt construction ─────────────────────────────────────────────────────
def format_question(question, choices):
    options = "\n".join(f"{LABELS[i]}. {c}" for i, c in enumerate(choices))
    return (
        "The following is a multiple choice question about law. "
        "Answer with the single best option.\n\n"
        f"{question}\n{options}"
    )

# ── 5. Logprob-based scoring (no generation, no parsing) ───────────────────────
@torch.no_grad()
def score(model, processor, text, letter_ids):
    """Return (predicted_letter, confidence) where confidence is the softmax
    probability of the chosen letter over the four options (0.25–1.0)."""
    inputs = processor(text=text, return_tensors="pt").to(model.device)
    logits = model(**inputs).logits[0, -1]              # logits for the next token
    letter_logits = logits[letter_ids]                  # restrict to A/B/C/D
    probs = torch.softmax(letter_logits.float(), dim=0)
    idx = int(torch.argmax(probs))
    return LABELS[idx], float(probs[idx])

def run_base(question, choices):
    text = format_question(question, choices) + "\nAnswer:"
    return score(base_model, base_processor, text, BASE_LETTER_IDS)

def run_it(question, choices):
    messages = [{"role": "user", "content": format_question(question, choices)}]
    text = it_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    text = text + "Answer:"                              # prime the letter position
    return score(it_model, it_processor, text, IT_LETTER_IDS)

# ── 6. Resume-safe output setup ────────────────────────────────────────────────
COLUMNS = [
    "id", "question", "choice_0", "choice_1", "choice_2", "choice_3", "answer",
    "gemma", "gemma_conf", "gemma_it", "gemma_it_conf",
]
if os.path.exists(OUTPUT_CSV):
    start_idx = len(pd.read_csv(OUTPUT_CSV))
    print(f"Resuming from row {start_idx}")
else:
    start_idx = 0
    pd.DataFrame(columns=COLUMNS).to_csv(OUTPUT_CSV, index=False)

# ── 7. Run inference ───────────────────────────────────────────────────────────
for _, row in tqdm(df.iloc[start_idx:].iterrows(), total=len(df) - start_idx, desc="Inference"):
    choices = [row["choice_0"], row["choice_1"], row["choice_2"], row["choice_3"]]

    gemma,    gemma_conf    = run_base(row["question"], choices)
    gemma_it, gemma_it_conf = run_it(row["question"],   choices)

    pd.DataFrame([{
        "id": row["id"], "question": row["question"],
        "choice_0": row["choice_0"], "choice_1": row["choice_1"],
        "choice_2": row["choice_2"], "choice_3": row["choice_3"],
        "answer": row["answer"],
        "gemma": gemma, "gemma_conf": round(gemma_conf, 4),
        "gemma_it": gemma_it, "gemma_it_conf": round(gemma_it_conf, 4),
    }]).to_csv(OUTPUT_CSV, mode="a", header=False, index=False)

print("Done →", OUTPUT_CSV)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

Inference:   0%|          | 0/1534 [00:00<?, ?it/s]

Done → FIXED_full_law_results.csv


In [2]:
"""
evaluate.py — Model evaluation script for multiple-choice QA results.

Usage:
    python evaluate.py results.csv

Expects columns: answer, gemma, gemma_conf, gemma_it, gemma_it_conf
NOTE: confidence columns are now the softmax probability of the chosen letter
(range ~0.25–1.0), NOT a 1–10 self-report.
Produces:  evaluation_report.txt  +  plots/*.png
"""

import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, ConfusionMatrixDisplay
)

# ── Config ────────────────────────────────────────────────────────────────────
LABELS      = ["A", "B", "C", "D"]
MODELS      = {"gemma": "Gemma (base)", "gemma_it": "Gemma-IT"}
CONF_COLS   = {"gemma": "gemma_conf",   "gemma_it": "gemma_it_conf"}
PALETTE     = {"gemma": "#4C72B0",      "gemma_it": "#DD8452"}
PLOTS_DIR   = "fixed_results/full_law_plots"
REPORT_FILE = "fixed_results/full_law_evaluation_report.txt"
INPUT_CSV = "FIXED_full_law_results.csv"

os.makedirs(PLOTS_DIR, exist_ok=True)

# ── Load ──────────────────────────────────────────────────────────────────────
csv_path = INPUT_CSV
df = pd.read_csv(csv_path)

# Normalise — drop rows where ground truth or either prediction is missing/invalid
df["answer"] = df["answer"].str.strip().str.upper()
for col in ["gemma", "gemma_it"]:
    df[col] = df[col].astype(str).str.strip().str.upper()

valid = df["answer"].isin(LABELS) & df["gemma"].isin(LABELS) & df["gemma_it"].isin(LABELS)
dropped = (~valid).sum()
df = df[valid].copy()
print(f"Loaded {len(df)} rows ({dropped} dropped — missing/invalid predictions).")

y_true = df["answer"]

lines = []  # collect report lines

def log(s=""):
    print(s)
    lines.append(s)

log("=" * 60)
log(f"Evaluation report — {csv_path}")
log(f"N = {len(df)}")
log("=" * 60)

# ══════════════════════════════════════════════════════════════════
# 1. ACCURACY
# ══════════════════════════════════════════════════════════════════
log("\n── Accuracy ──────────────────────────────────────────────────")
for key, label in MODELS.items():
    acc = (df[key] == y_true).mean()
    log(f"  {label:20s}  {acc:.4f}  ({int(acc * len(df))}/{len(df)})")

# ══════════════════════════════════════════════════════════════════
# 2. CLASSIFICATION REPORT (precision / recall / F1 per class)
# ══════════════════════════════════════════════════════════════════
log("\n── Classification Report ─────────────────────────────────────")
for key, label in MODELS.items():
    log(f"\n  {label}")
    report = classification_report(y_true, df[key], labels=LABELS, zero_division=0)
    for line in report.splitlines():
        log("    " + line)

# ══════════════════════════════════════════════════════════════════
# 3. CONFUSION MATRICES
# ══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (key, label) in zip(axes, MODELS.items()):
    cm = confusion_matrix(y_true, df[key], labels=LABELS)
    disp = ConfusionMatrixDisplay(cm, display_labels=LABELS)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(label, fontsize=13)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
plt.suptitle("Confusion Matrices", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.close()
log(f"\n  → {PLOTS_DIR}/confusion_matrices.png")

# ══════════════════════════════════════════════════════════════════
# 4. TPR / FPR per class (one-vs-rest)
# ══════════════════════════════════════════════════════════════════
log("\n── TPR / FPR per class (one-vs-rest) ────────────────────────")
log(f"  {'Class':<6}  {'Model':<22}  {'TPR':>6}  {'FPR':>6}  {'TNR':>6}  {'FNR':>6}")
log(f"  {'-'*5}  {'-'*22}  {'-'*6}  {'-'*6}  {'-'*6}  {'-'*6}")

for cls in LABELS:
    for key, label in MODELS.items():
        tp = ((y_true == cls) & (df[key] == cls)).sum()
        fn = ((y_true == cls) & (df[key] != cls)).sum()
        fp = ((y_true != cls) & (df[key] == cls)).sum()
        tn = ((y_true != cls) & (df[key] != cls)).sum()
        tpr = tp / (tp + fn) if (tp + fn) else 0
        fpr = fp / (fp + tn) if (fp + tn) else 0
        tnr = tn / (fp + tn) if (fp + tn) else 0
        fnr = fn / (tp + fn) if (tp + fn) else 0
        log(f"  {cls:<6}  {label:<22}  {tpr:>6.3f}  {fpr:>6.3f}  {tnr:>6.3f}  {fnr:>6.3f}")

# ══════════════════════════════════════════════════════════════════
# 5. ROC CURVES (one-vs-rest, using confidence as score)
# ══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (key, label) in zip(axes, MODELS.items()):
    conf_col = CONF_COLS[key]
    conf_valid = df[conf_col].notna()
    df_c = df[conf_valid].copy()
    df_c[conf_col] = pd.to_numeric(df_c[conf_col], errors="coerce")
    df_c = df_c[df_c[conf_col].notna()]

    for cls in LABELS:
        y_bin   = (df_c["answer"] == cls).astype(int)
        # Use confidence (already 0–1) when the model picked this class, 0 otherwise
        scores  = np.where(df_c[key] == cls, df_c[conf_col], 0.0)
        if y_bin.sum() == 0:
            continue
        fpr_r, tpr_r, _ = roc_curve(y_bin, scores)
        roc_auc = auc(fpr_r, tpr_r)
        ax.plot(fpr_r, tpr_r, label=f"{cls} (AUC={roc_auc:.2f})")

    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC — {label}")
    ax.legend(fontsize=8)

plt.suptitle("ROC Curves (one-vs-rest)", fontsize=14)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/roc_curves.png", dpi=150, bbox_inches="tight")
plt.close()
log(f"\n  → {PLOTS_DIR}/roc_curves.png")

# ══════════════════════════════════════════════════════════════════
# 6. CALIBRATION PLOTS (reliability diagrams)
# ══════════════════════════════════════════════════════════════════
def calibration_bins(df, pred_col, conf_col, n_bins=10):
    d = df[[pred_col, conf_col, "answer"]].copy()
    d[conf_col] = pd.to_numeric(d[conf_col], errors="coerce")
    d = d.dropna(subset=[conf_col])
    d["correct"] = (d[pred_col] == d["answer"]).astype(float)
    d["conf_norm"] = d[conf_col]   # already 0–1
    bins = np.linspace(0, 1, n_bins + 1)
    bin_acc, bin_conf, bin_n = [], [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (d["conf_norm"] >= lo) & (d["conf_norm"] < hi)
        if mask.sum() == 0:
            continue
        bin_acc.append(d.loc[mask, "correct"].mean())
        bin_conf.append(d.loc[mask, "conf_norm"].mean())
        bin_n.append(mask.sum())
    return np.array(bin_conf), np.array(bin_acc), np.array(bin_n)

def ece(bin_conf, bin_acc, bin_n):
    total = bin_n.sum()
    return np.sum(bin_n / total * np.abs(bin_acc - bin_conf))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
log("\n── Calibration (ECE) ─────────────────────────────────────────")
for ax, (key, label) in zip(axes, MODELS.items()):
    conf_col = CONF_COLS[key]
    bc, ba, bn = calibration_bins(df, key, conf_col)
    e = ece(bc, ba, bn)
    log(f"  {label:22s}  ECE = {e:.4f}")

    ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="Perfect calibration")
    ax.bar(bc, ba, width=0.08, alpha=0.6, color=PALETTE[key], label="Accuracy")
    ax.plot(bc, bc, "k--", lw=0.8)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("Mean confidence")
    ax.set_ylabel("Fraction correct")
    ax.set_title(f"Calibration — {label}\nECE = {e:.4f}")
    ax.legend(fontsize=8)

plt.suptitle("Reliability Diagrams", fontsize=14)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/calibration.png", dpi=150, bbox_inches="tight")
plt.close()
log(f"\n  → {PLOTS_DIR}/calibration.png")

# ══════════════════════════════════════════════════════════════════
# 7. CONFIDENCE DISTRIBUTIONS (correct vs incorrect)
# ══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (key, label) in zip(axes, MODELS.items()):
    conf_col = CONF_COLS[key]
    d = df[[key, conf_col, "answer"]].copy()
    d[conf_col] = pd.to_numeric(d[conf_col], errors="coerce")
    d = d.dropna(subset=[conf_col])
    correct   = d[d[key] == d["answer"]][conf_col]
    incorrect = d[d[key] != d["answer"]][conf_col]

    bins = np.linspace(0, 1, 11)   # confidence is a probability 0–1
    ax.hist(correct,   bins=bins, alpha=0.6, color="steelblue", label=f"Correct (n={len(correct)})",   density=True)
    ax.hist(incorrect, bins=bins, alpha=0.6, color="tomato",    label=f"Incorrect (n={len(incorrect)})", density=True)
    ax.set_xlabel("Confidence  P(chosen letter)")
    ax.set_ylabel("Density")
    ax.set_title(label)
    ax.legend(fontsize=8)

plt.suptitle("Confidence Distribution: Correct vs Incorrect", fontsize=14)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/confidence_distribution.png", dpi=150, bbox_inches="tight")
plt.close()
log(f"\n  → {PLOTS_DIR}/confidence_distribution.png")

# ══════════════════════════════════════════════════════════════════
# 8. ACCURACY BY CONFIDENCE LEVEL
# ══════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(9, 4))
conf_bins = np.linspace(0, 1, 11)   # bin the continuous probability into deciles
for key, label in MODELS.items():
    conf_col = CONF_COLS[key]
    d = df[[key, conf_col, "answer"]].copy()
    d[conf_col] = pd.to_numeric(d[conf_col], errors="coerce")
    d = d.dropna(subset=[conf_col])
    d["correct"] = (d[key] == d["answer"]).astype(float)
    d["conf_bin"] = pd.cut(d[conf_col], bins=conf_bins, include_lowest=True)
    grouped = d.groupby("conf_bin", observed=True)["correct"].agg(["mean", "count"]).reset_index()
    centers = grouped["conf_bin"].apply(lambda b: b.mid).astype(float)
    ax.plot(centers, grouped["mean"], marker="o", label=label, color=PALETTE[key])

ax.axhline(0.25, color="grey", linestyle="--", lw=0.8, label="Random baseline (4-class)")
ax.set_xlabel("Confidence  P(chosen letter)")
ax.set_ylabel("Accuracy")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Accuracy by Confidence Level")
ax.legend()
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/accuracy_by_confidence.png", dpi=150, bbox_inches="tight")
plt.close()
log(f"\n  → {PLOTS_DIR}/accuracy_by_confidence.png")

# ══════════════════════════════════════════════════════════════════
# 9. MODEL AGREEMENT
# ══════════════════════════════════════════════════════════════════
agree     = (df["gemma"] == df["gemma_it"])
both_right = agree & (df["gemma"] == y_true)
both_wrong = agree & (df["gemma"] != y_true)
disagree  = ~agree

log("\n── Model Agreement ───────────────────────────────────────────")
log(f"  Agree (both correct):   {both_right.sum():4d}  ({both_right.mean():.2%})")
log(f"  Agree (both wrong):     {both_wrong.sum():4d}  ({both_wrong.mean():.2%})")
log(f"  Disagree:               {disagree.sum():4d}  ({disagree.mean():.2%})")

# When they disagree, who is right?
if disagree.sum() > 0:
    base_right_it_wrong = (disagree & (df["gemma"]    == y_true)).sum()
    it_right_base_wrong = (disagree & (df["gemma_it"] == y_true)).sum()
    neither             = (disagree & (df["gemma"] != y_true) & (df["gemma_it"] != y_true)).sum()
    log(f"  On disagreements — base correct: {base_right_it_wrong}, IT correct: {it_right_base_wrong}, neither: {neither}")

# ══════════════════════════════════════════════════════════════════
# 10. ANSWER DISTRIBUTION
# ══════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
sources = {"Ground Truth": y_true, "Gemma (base)": df["gemma"], "Gemma-IT": df["gemma_it"]}
for ax, (title, series) in zip(axes, sources.items()):
    counts = series.value_counts().reindex(LABELS, fill_value=0)
    ax.bar(LABELS, counts.values, color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"])
    ax.set_title(title)
    ax.set_ylabel("Count")
    ax.set_ylim(0, len(df))
    for i, v in enumerate(counts.values):
        ax.text(i, v + 1, str(v), ha="center", fontsize=9)

plt.suptitle("Answer Distribution", fontsize=14)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/answer_distribution.png", dpi=150, bbox_inches="tight")
plt.close()
log(f"\n  → {PLOTS_DIR}/answer_distribution.png")

# ══════════════════════════════════════════════════════════════════
# Write report
# ══════════════════════════════════════════════════════════════════
log("\n" + "=" * 60)
with open(REPORT_FILE, "w") as f:
    f.write("\n".join(lines))
print(f"\nReport saved → {REPORT_FILE}")
print(f"Plots saved  → {PLOTS_DIR}/")

Loaded 1534 rows (0 dropped — missing/invalid predictions).
Evaluation report — FIXED_full_law_results.csv
N = 1534

── Accuracy ──────────────────────────────────────────────────
  Gemma (base)          0.4094  (628/1534)
  Gemma-IT              0.4563  (700/1534)

── Classification Report ─────────────────────────────────────

  Gemma (base)
                  precision    recall  f1-score   support
    
               A       0.36      0.37      0.37       377
               B       0.40      0.34      0.37       367
               C       0.43      0.48      0.45       415
               D       0.44      0.44      0.44       375
    
        accuracy                           0.41      1534
       macro avg       0.41      0.41      0.41      1534
    weighted avg       0.41      0.41      0.41      1534

  Gemma-IT
                  precision    recall  f1-score   support
    
               A       0.42      0.43      0.43       377
               B       0.44      0.49      0.47